# Render the seven-point videos with a skeleton

This uses the finished `mobile_fast_20260925_040943` run and its best checkpoint at epoch 110. The predictions already exist, so this only redraws the videos. The sample cell renders Test 30 at a 0.2 display cutoff to check whether lower sensitivity removes brief dropouts; it does not alter predictions or retrain the model. Run the install cell only if DeepLabCut is not already installed in this Colab runtime. A GPU is not required. The old videos stay untouched.


In [ ]:
%pip -q install --pre deeplabcut


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path, PurePosixPath
import shutil, zipfile, yaml, deeplabcut
import pandas as pd
from deeplabcut.utils import auxiliaryfunctions

ROOT = Path('/content/drive/MyDrive/sp1DLC')
RUN = ROOT / 'Colab_Training_Runs/mobile_fast_20260925_040943'
PREDICTIONS = RUN / 'videos'
SOURCE_ZIP = ROOT / 'all_32_videos.zip'
OUTPUT = RUN / 'Seven_Point_Videos'
WORK = Path('/content/seven_point_render')
VIEW = Path('/content/seven_point_model')
FOCUS = ['nose', 'left_ear', 'right_ear', 'spine_mid', 'left_hip', 'right_hip', 'tail_base']
SKELETON = [['nose', 'left_ear'], ['nose', 'right_ear'], ['left_ear', 'spine_mid'], ['right_ear', 'spine_mid'], ['spine_mid', 'left_hip'], ['spine_mid', 'right_hip'], ['left_hip', 'tail_base'], ['right_hip', 'tail_base']]

assert SOURCE_ZIP.is_file(), f'Missing original video archive: {SOURCE_ZIP}'
assert (RUN / 'dlc-models-pytorch').is_dir(), f'Missing trained model: {RUN}'
assert list((RUN / 'dlc-models-pytorch').rglob('snapshot-best-110.pt')), 'Best epoch-110 checkpoint is missing.'
h5_files = sorted(PREDICTIONS.glob('*snapshot_best-110.h5'))
assert len(h5_files) == 31, f'Expected 31 prediction files, found {len(h5_files)}'
expected = {h5.name.partition('DLC_')[0] + '.mp4' for h5 in h5_files}
assert len(expected) == 31, 'Could not match predictions to 31 original videos.'

VIDEO_ZIP = Path('/content/all_32_videos.zip')
if not VIDEO_ZIP.is_file() or VIDEO_ZIP.stat().st_size != SOURCE_ZIP.stat().st_size:
    shutil.copy2(SOURCE_ZIP, VIDEO_ZIP)
WORK.mkdir(exist_ok=True)
videos = {}
with zipfile.ZipFile(VIDEO_ZIP) as archive:
    assert archive.testzip() is None, 'Original video ZIP failed its integrity check.'
    for info in archive.infolist():
        name = PurePosixPath(info.filename.replace('\\', '/')).name
        if name in expected:
            assert name not in videos, f'Duplicate video in archive: {name}'
            target = WORK / name
            if not target.is_file() or target.stat().st_size != info.file_size:
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
            videos[name] = target
assert set(videos) == expected, f'Original videos missing: {sorted(expected - set(videos))}'

for h5 in h5_files:
    meta = h5.with_name(h5.stem + '_meta.pickle')
    assert meta.is_file(), f'Missing prediction metadata: {meta.name}'
    filtered_h5 = WORK / h5.name
    predictions = pd.read_hdf(h5)
    present = list(predictions.columns.get_level_values('bodyparts').unique())
    assert all(point in present for point in FOCUS), f'{h5.name} is missing one of the seven focus points.'
    predictions = predictions.loc[:, predictions.columns.get_level_values('bodyparts').isin(FOCUS)]
    assert list(predictions.columns.get_level_values('bodyparts').unique()) == FOCUS, f'Unexpected bodypart order in {h5.name}'
    predictions.to_hdf(filtered_h5, key='df_with_missing', mode='w')
    target_meta = WORK / meta.name
    if not target_meta.is_file() or target_meta.stat().st_size != meta.stat().st_size:
        shutil.copy2(meta, target_meta)

VIEW.mkdir(exist_ok=True)
models_link = VIEW / 'dlc-models-pytorch'
if not models_link.is_symlink():
    assert not models_link.exists(), f'Unexpected model folder: {models_link}'
    models_link.symlink_to(RUN / 'dlc-models-pytorch', target_is_directory=True)
with (RUN / 'config.yaml').open(encoding='utf-8') as source:
    cfg = yaml.safe_load(source)
assert all(point in cfg['bodyparts'] for point in FOCUS), 'Seven-point list does not match the DLC config.'
cfg['project_path'] = str(VIEW)
cfg['bodyparts'] = FOCUS
cfg['snapshotindex'] = -1
cfg['dotsize'] = 3
cfg['skeleton'] = SKELETON
CONFIG = VIEW / 'config.yaml'
with CONFIG.open('w', encoding='utf-8') as output:
    yaml.safe_dump(cfg, output, sort_keys=False)
scorer, _ = auxiliaryfunctions.get_scorer_name(cfg, 1, cfg['TrainingFraction'][0], snapshot_index='best')
assert all(h5.name == h5.name.partition('DLC_')[0] + scorer + '.h5' for h5 in h5_files), f'Prediction scorer does not match best checkpoint: {scorer}'
OUTPUT.mkdir(exist_ok=True)
print('Ready to re-render', len(videos), 'original videos using only:', ', '.join(FOCUS))
print('Output folder:', OUTPUT)


In [ ]:
sample = videos['Test 30.mp4']
result = deeplabcut.create_labeled_video(str(CONFIG), [str(sample)], destfolder=str(WORK), shuffle=1, displayedbodyparts=FOCUS, snapshot_index='best', pcutoff=0.2, draw_skeleton=True, trailpoints=0, fastmode=True, plot_bboxes=False, max_workers=1, overwrite=True)
assert result and result[0], f'Failed to render {sample.name}'
sample_video = WORK / f'Test 30{scorer}_p20_labeled.mp4'
assert sample_video.is_file() and sample_video.stat().st_size > 0, 'Sample video was not created.'
shutil.copy2(sample_video, OUTPUT / sample_video.name)
print('Seven-point sample saved to Drive:', OUTPUT / sample_video.name)
from IPython.display import Video, display
display(Video(str(sample_video), embed=True, width=640))


In [ ]:
remaining = [str(path) for name, path in sorted(videos.items()) if name != 'Test 11.mp4']
results = deeplabcut.create_labeled_video(str(CONFIG), remaining, destfolder=str(WORK), shuffle=1, displayedbodyparts=FOCUS, snapshot_index='best', pcutoff=0.3, draw_skeleton=True, trailpoints=0, fastmode=True, plot_bboxes=False, max_workers=1, overwrite=True)
assert len(results) == len(remaining) and all(results), 'One or more videos failed; check the output above.'
made = sorted(WORK.glob(f'*{scorer}_p30_labeled.mp4'))
assert len(made) == 31, f'Expected 31 seven-point videos, found {len(made)}'
for video in made:
    destination = OUTPUT / video.name
    shutil.copy2(video, destination)
assert len(list(OUTPUT.glob(f'*{scorer}_p30_labeled.mp4'))) == 31
print('Finished: 31 seven-point MP4s in', OUTPUT)


Only the seven listed points and their connecting skeleton appear on these overlays. The notebook makes temporary seven-column H5 copies in `/content/seven_point_render` before drawing, so the renderer has no non-focus bodyparts available. The new `_p30_labeled.mp4` files are written to `Seven_Point_Videos`; existing `_p50_labeled.mp4` files, older videos, and original 19-point prediction H5 files remain untouched. The older 0.1 render remains available for checking lower-confidence point locations.
